# Preprocessing on Weather Dataset:

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, last, count, to_timestamp
from pyspark.sql.functions import hour, substring, to_date, date_format, avg
from datetime import datetime, timedelta
from pyspark.sql import Window
from pyspark.sql.functions import row_number
from functools import reduce
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("preprocessing_weather")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.executor.heartbeatInterval", "30s")
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/16 14:25:43 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.12.218.41 instead (on interface en0)
24/08/16 14:25:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/16 14:25:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read Weather File:

In [3]:
base_dir = "../data"
weather_path = base_dir + '/raw/weather_data/'
weather_sdf = spark.read.parquet(weather_path)
weather_sdf.show(5)

+-----------+-------------------+--------+---------+---------+--------------------+-----------+------+----------------------+-------------------------+------------------------+-------------------+------------------------+--------------------+----------------------+----------------------+--------------------+----------------------+---------------------+----------------+------------------------+-------------------+-------------------+---------------+-------+------+-------------------------------+------------------------------+----------------------------+----------------------------+---------------------------+------------------------------+---------------------+----------------------+------------------------------------------+----------------------+------------------------------+------------------------------+----------------------+------------------+------------------+--------------+-------------+---------------------------+-----------------------+------------+----------------+--------

In [4]:
weather_sdf.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- NAME: string (nullable = true)
 |-- REPORT_TYPE: string (nullable = true)
 |-- SOURCE: double (nullable = true)
 |-- HourlyAltimeterSetting: double (nullable = true)
 |-- HourlyDewPointTemperature: double (nullable = true)
 |-- HourlyDryBulbTemperature: double (nullable = true)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- HourlyPresentWeatherType: string (nullable = true)
 |-- HourlyPressureChange: double (nullable = true)
 |-- HourlyPressureTendency: double (nullable = true)
 |-- HourlyRelativeHumidity: double (nullable = true)
 |-- HourlySkyConditions: string (nullable = true)
 |-- HourlySeaLevelPressure: double (nullable = true)
 |-- HourlyStationPressure: double (nullable = true)
 |-- HourlyVisibility: string (nullable = true)
 |-- HourlyWetBulbTemperature: double (nu

In [5]:
# Check the shape of parquet file
num_rows = weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 11803
Number of columns: 125


# Drop Unrelated Rows and Columns:

Since the timeline for our research is 2023-07 to 2023-12, we remove the rows from January to June.

In [6]:
weather_sdf = weather_sdf.filter(~col('DATE').startswith('2023-01') & 
                                 ~col('DATE').startswith('2023-02') &
                                 ~col('DATE').startswith('2023-03') &
                                 ~col('DATE').startswith('2023-04') &
                                 ~col('DATE').startswith('2023-05') &
                                 ~col('DATE').startswith('2023-06'))
weather_sdf.show(5)

+-----------+-------------------+--------+---------+---------+--------------------+-----------+------+----------------------+-------------------------+------------------------+-------------------+------------------------+--------------------+----------------------+----------------------+-------------------+----------------------+---------------------+----------------+------------------------+-------------------+-------------------+---------------+-------+------+-------------------------------+------------------------------+----------------------------+----------------------------+---------------------------+------------------------------+---------------------+----------------------+------------------------------------------+----------------------+------------------------------+------------------------------+----------------------+------------------+------------------+--------------+-------------+---------------------------+-----------------------+------------+----------------+---------

Leave only the helpful columns:

In [7]:
hourly_weather_sdf = weather_sdf.select('DATE',
                                        'HourlyDryBulbTemperature', 
                                        'HourlyPrecipitation',
                                        'HourlyVisibility', 
                                        'HourlyWindSpeed')
hourly_weather_sdf.show(5)

+-------------------+------------------------+-------------------+----------------+---------------+
|               DATE|HourlyDryBulbTemperature|HourlyPrecipitation|HourlyVisibility|HourlyWindSpeed|
+-------------------+------------------------+-------------------+----------------+---------------+
|2023-07-01T00:00:00|                    NULL|               NULL|            NULL|           NULL|
|2023-07-01T00:00:00|                    NULL|               NULL|            NULL|           NULL|
|2023-07-01T00:51:00|                    22.2|                0.0|           9.656|            0.0|
|2023-07-01T01:51:00|                    21.7|                0.0|           9.656|            2.6|
|2023-07-01T02:51:00|                    21.1|                0.0|           9.656|            1.5|
+-------------------+------------------------+-------------------+----------------+---------------+
only showing top 5 rows



In [8]:
# Check the shape of parquet file
num_rows = hourly_weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 6140
Number of columns: 5


In [9]:
# Add new columns `DateOnly` and `Hour``, then drop the original 'DATE' column
hourly_weather_sdf = hourly_weather_sdf.withColumn("DATE", to_timestamp(col("DATE"), "yyyy-MM-dd'T'HH:mm:ss"))
hourly_weather_sdf = hourly_weather_sdf.withColumn("DateOnly", date_format(col("DATE"), "yyyy-MM-dd"))
hourly_weather_sdf = hourly_weather_sdf.withColumn("Hour", hour(col("DATE")))
hourly_weather_sdf = hourly_weather_sdf.drop("DATE")
hourly_weather_sdf.show(5)

+------------------------+-------------------+----------------+---------------+----------+----+
|HourlyDryBulbTemperature|HourlyPrecipitation|HourlyVisibility|HourlyWindSpeed|  DateOnly|Hour|
+------------------------+-------------------+----------------+---------------+----------+----+
|                    NULL|               NULL|            NULL|           NULL|2023-07-01|   0|
|                    NULL|               NULL|            NULL|           NULL|2023-07-01|   0|
|                    22.2|                0.0|           9.656|            0.0|2023-07-01|   0|
|                    21.7|                0.0|           9.656|            2.6|2023-07-01|   1|
|                    21.1|                0.0|           9.656|            1.5|2023-07-01|   2|
+------------------------+-------------------+----------------+---------------+----------+----+
only showing top 5 rows



# Imputation of Data:

### Replace "T" with 0:

According to National Center of Envirmental Information, "T" usually stands for "Trace", which means trace precipitation. This means the amount of precipitation was so small that it was not measurable, but it still occurred. Typically, trace precipitation is less than 0.01 inches. So, we substitute "T" with the number 0 for convience.

In [10]:
hourly_weather_sdf = hourly_weather_sdf.withColumn("HourlyPrecipitation",
                                                 when(col("HourlyPrecipitation") == "T", 0.0).otherwise(col("HourlyPrecipitation"))
                                                 ).withColumn("HourlyVisibility",
                                                              when(col("HourlyVisibility") == "T", 0.0).otherwise(col("HourlyVisibility")))
hourly_weather_sdf.show(5)

+------------------------+-------------------+----------------+---------------+----------+----+
|HourlyDryBulbTemperature|HourlyPrecipitation|HourlyVisibility|HourlyWindSpeed|  DateOnly|Hour|
+------------------------+-------------------+----------------+---------------+----------+----+
|                    NULL|               NULL|            NULL|           NULL|2023-07-01|   0|
|                    NULL|               NULL|            NULL|           NULL|2023-07-01|   0|
|                    22.2|                0.0|           9.656|            0.0|2023-07-01|   0|
|                    21.7|                0.0|           9.656|            2.6|2023-07-01|   1|
|                    21.1|                0.0|           9.656|            1.5|2023-07-01|   2|
+------------------------+-------------------+----------------+---------------+----------+----+
only showing top 5 rows



### Replace NULLs with the previous effective value:

Since the hourly weather data is the time series data, we decide to impute NULL with its previous effective value:

In [11]:
columns_to_impute = ["HourlyDryBulbTemperature", 
                     "HourlyPrecipitation", 
                     "HourlyVisibility", 
                     "HourlyWindSpeed"]

window_spec = Window.partitionBy("Hour").orderBy("DateOnly").rowsBetween(Window.unboundedPreceding, Window.currentRow)

for column in columns_to_impute:
    hourly_weather_sdf = hourly_weather_sdf.withColumn(
        column,
        last(col(column), ignorenulls=True).over(window_spec)
    )

In [12]:
# Check the shape of parquet file
num_rows = hourly_weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 6140
Number of columns: 6


### Remove the rows with NULLs except `DATE` column:

In [13]:
# Calculate the amount of NULL in each column
null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in hourly_weather_sdf.columns]
na_counts = hourly_weather_sdf.agg(*null_counts_expr)
na_counts.show()

+------------------------+-------------------+----------------+---------------+--------+----+
|HourlyDryBulbTemperature|HourlyPrecipitation|HourlyVisibility|HourlyWindSpeed|DateOnly|Hour|
+------------------------+-------------------+----------------+---------------+--------+----+
|                       2|                  2|               2|              4|       0|   0|
+------------------------+-------------------+----------------+---------------+--------+----+



Maybe there is no previous effective value for these NULLs, so I delete them.

In [14]:
# Then we delete the rows which contain NULLs except for `DATE` column:
hourly_weather_sdf = hourly_weather_sdf.dropna(subset=columns_to_impute)

In [15]:
# Check the shape of parquet file
num_rows = hourly_weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = hourly_weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 6136
Number of columns: 6


### Impute missing data:

I have checked that some dates' data are missing, so we impute them by its previous data due to the weather dataset is time series.

Following are the main points for the next code chunk:
- Generate the full date range: create a list of all dates within the specified range
- Generate all Hours for each date: create a DataFrame with all hours for each date
- Perform the left join: join this DataFrame with your existing data to ensure all possible timestamps are included
- Fill missing data: use forward fill to handle any missing values

In [16]:
# Define the start and end dates
start_date = datetime.strptime("2023-07-01", "%Y-%m-%d")
end_date = datetime.strptime("2023-12-31", "%Y-%m-%d")

# Generate all dates between start_date and end_date
date_range = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d") for i in range((end_date - start_date).days + 1)]

# Create a DataFrame with all dates and hours
dates_df = spark.createDataFrame([(date,) for date in date_range], ["DateOnly"])
hours = [(hour,) for hour in range(24)]
hours_df = spark.createDataFrame(hours, ["Hour"])

# Create all combinations of dates and hours
all_dates_hours_df = dates_df.crossJoin(hours_df)

# Perform left join to get all timestamps with existing data
full_hourly_weather_sdf = all_dates_hours_df.join(hourly_weather_sdf, on=["DateOnly", "Hour"], how="left")

# Define the window specification for forward fill
window_spec = Window.partitionBy("DateOnly").orderBy("Hour").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Forward fill for each column that needs it
for column in columns_to_impute:
    full_hourly_weather_sdf = full_hourly_weather_sdf.withColumn(
        column,
        F.last(F.col(column), ignorenulls=True).over(window_spec)
    )

In [17]:
num_rows = full_hourly_weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = full_hourly_weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 6144
Number of columns: 6


In [18]:
avg_hourly_weather_sdf = full_hourly_weather_sdf.groupBy("DateOnly", "Hour").agg(
    F.avg("HourlyDryBulbTemperature").alias("AvgHourlyTemp"),
    F.avg("HourlyPrecipitation").alias("AvgHourlyPrecipitation"),
    F.avg("HourlyVisibility").alias("AvgHourlyVisibility"),
    F.avg("HourlyWindSpeed").alias("AvgHourlyWindSpeed")
)


### Check whether the number of rows is correct:

In [19]:
print("Correct number of rows:", (31+31+30+31+30+31)*24) 

Correct number of rows: 4416


In [20]:
num_rows = avg_hourly_weather_sdf.count()
print(f"Number of rows: {num_rows}")

columns = avg_hourly_weather_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

Number of rows: 4416
Number of columns: 6


### Check whether there is NULL value:

In [21]:
# Calculate the amount of NULL in each column
null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in avg_hourly_weather_sdf.columns]
na_counts = avg_hourly_weather_sdf.agg(*null_counts_expr)
na_counts.show()

+--------+----+-------------+----------------------+-------------------+------------------+
|DateOnly|Hour|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHourlyWindSpeed|
+--------+----+-------------+----------------------+-------------------+------------------+
|       0|   0|            0|                     0|                  2|                 0|
+--------+----+-------------+----------------------+-------------------+------------------+



I have no idea why there still exists NULLs, so I impute zero.

In [22]:
avg_hourly_weather_sdf = avg_hourly_weather_sdf.fillna({"AvgHourlyVisibility": 0})

# Calculate the amount of NULL in each column
null_counts_expr = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in avg_hourly_weather_sdf.columns]
na_counts = avg_hourly_weather_sdf.agg(*null_counts_expr)
na_counts.show()

+--------+----+-------------+----------------------+-------------------+------------------+
|DateOnly|Hour|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHourlyWindSpeed|
+--------+----+-------------+----------------------+-------------------+------------------+
|       0|   0|            0|                     0|                  0|                 0|
+--------+----+-------------+----------------------+-------------------+------------------+



In [23]:
avg_hourly_weather_sdf.show(5)

+----------+----+-------------+----------------------+-------------------+------------------+
|  DateOnly|Hour|AvgHourlyTemp|AvgHourlyPrecipitation|AvgHourlyVisibility|AvgHourlyWindSpeed|
+----------+----+-------------+----------------------+-------------------+------------------+
|2023-07-01|   0|         22.2|                   0.0|              9.656|               0.0|
|2023-07-01|   1|         21.7|                   0.0|              9.656|               2.6|
|2023-07-01|   2|         21.1|                   0.0|              9.656|               1.5|
|2023-07-01|   3|         21.1|                   0.0|             11.265|               1.5|
|2023-07-01|   4|         20.6|                   0.0|              9.656|               0.0|
+----------+----+-------------+----------------------+-------------------+------------------+
only showing top 5 rows



In [24]:
avg_hourly_weather_sdf.printSchema()

root
 |-- DateOnly: string (nullable = true)
 |-- Hour: long (nullable = true)
 |-- AvgHourlyTemp: double (nullable = true)
 |-- AvgHourlyPrecipitation: double (nullable = true)
 |-- AvgHourlyVisibility: double (nullable = false)
 |-- AvgHourlyWindSpeed: double (nullable = true)



# Save the Preprocessed Weather Dataset:

In [25]:
weather_dir = base_dir + '/curated/weather_data'
file_name = 'preprocessed_hourly_weather'
weather_path = os.path.join(weather_dir, file_name)
avg_hourly_weather_sdf.write.mode('overwrite').parquet(weather_path)